# Figure 7 setup — PRJNA671738 analyze (UHVDB r6, no genecoverage)

Operational notebook to prepare and monitor SRA profiling of BioProject **PRJNA671738** against UHVDB r6 species reps.

Pipeline (per accession, via `run_analyze.sbatch`):
`sracha → fastp → deacon → sylph → CoverM depth`  
Gene coverage is intentionally skipped (r6 annotations not ready).

**Steps**
1. Fetch SRA run accessions → write `samples.tsv`
2. Check required paths
3. Submit `prepare_species_reps.sbatch`, then `./submit_analyze.sh`
4. Inventory `results/` for completed profiles / depths


In [ ]:
from pathlib import Path
import csv
import json
import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

OUT = Path("/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript-update/figure_7")
BASE = Path("/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB")
BIOPROJECT = "PRJNA671738"
METADATA = BASE / "uhvdb-manuscript-update/figure_1/uhvdb_v6_metadata_sra.tsv"
ANICLUSTER = BASE / "uhvdb-manuscript-update/figure_1/results_backup/anicluster"
TOOLKIT = BASE / "toolkit2"

SAMPLES_TSV = OUT / "samples.tsv"
print("OUT:", OUT)
assert OUT.is_dir()


## 1. Resolve SRA accessions for PRJNA671738

Fetch all SRA run accessions via NCBI E-utilities, write `samples.tsv`, and cross-check against UHVDB r6 metadata.


In [ ]:
def _get(url, retries=5, sleep_s=1.0):
    last_err = None
    for i in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=60) as resp:
                return resp.read()
        except Exception as e:
            last_err = e
            time.sleep(sleep_s * (i + 1))
    raise RuntimeError(f"Failed GET {url}: {last_err}")


def fetch_sra_runs_for_bioproject(bioproject):
    """Return sorted unique SRA run accessions (SRR/ERR/DRR) for a BioProject."""
    term = urllib.parse.quote(f"{bioproject}[BioProject]")
    search_url = (
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
        f"?db=sra&term={term}&retmax=100000&retmode=json"
    )
    data = json.loads(_get(search_url))
    ids = data.get("esearchresult", {}).get("idlist", [])
    if not ids:
        raise RuntimeError(f"No SRA UIDs found for {bioproject}")

    runs = set()
    batch = 200
    for start in range(0, len(ids), batch):
        chunk = ids[start : start + batch]
        fetch_url = (
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
            f"?db=sra&id={','.join(chunk)}&retmode=xml"
        )
        xml_bytes = _get(fetch_url)
        root = ET.fromstring(xml_bytes)
        for run in root.iter("RUN"):
            acc = run.attrib.get("accession")
            if acc and acc.startswith(("SRR", "ERR", "DRR")):
                runs.add(acc)
        time.sleep(0.34)

    if not runs:
        raise RuntimeError(f"Parsed 0 run accessions from SRA XML for {bioproject}")
    return sorted(runs)


ncbi_runs = fetch_sra_runs_for_bioproject(BIOPROJECT)
print(f"NCBI SRA runs for {BIOPROJECT}: {len(ncbi_runs)}")
print("first 10:", ncbi_runs[:10])


In [ ]:
# Cross-check vs UHVDB r6 metadata
meta_accs = set()
with METADATA.open() as fh:
    reader = csv.DictReader(fh, delimiter="\t")
    for row in reader:
        if row.get("bioproject") == BIOPROJECT:
            acc = (row.get("acc") or "").strip()
            if acc:
                meta_accs.add(acc)

ncbi_set = set(ncbi_runs)
print(f"Metadata unique accs for {BIOPROJECT}: {len(meta_accs)}")
print(f"In NCBI only: {len(ncbi_set - meta_accs)}")
print(f"In metadata only: {len(meta_accs - ncbi_set)}")
print(f"Intersection: {len(ncbi_set & meta_accs)}")

# Prefer full BioProject set from NCBI
runs = ncbi_runs
with SAMPLES_TSV.open("w", newline="") as fh:
    w = csv.writer(fh, delimiter="\t", lineterminator="\n")
    w.writerow(["sample", "acc"])
    for acc in runs:
        w.writerow([acc, acc])

print(f"Wrote {SAMPLES_TSV} ({len(runs)} samples)")
print("\n".join(SAMPLES_TSV.read_text().splitlines()[:5]))


## 2. Path sanity checks


In [ ]:
paths = {
    "new_reps": ANICLUSTER / "new_genomovar_reps.new_reps.fna.gz",
    "old_reps": ANICLUSTER / "new_genomovar_reps.old_reps.fna.gz",
    "deacon_idx": TOOLKIT / "databases/deacon/0.13.2/panhuman-1.k31w15.idx",
    "bacteria_syldb": TOOLKIT / "databases/sylph/v0.3-c1000-gtdb-r214.syldb",
    "analyze_img": TOOLKIT
    / "databases/.singularity-cache"
    / "community-cr-prod.seqera.io-docker-registry-v2-blobs-sha256-f7-f71fcf13109179464415b2307157da066c9b68aee280313cd3265cb49eb83f0b-data.img",
    "sylph_img": TOOLKIT
    / "databases/.singularity-cache"
    / "depot.galaxyproject.org-singularity-sylph-0.9.0--ha6fb395_0.img",
    "samples_tsv": SAMPLES_TSV,
    "species_reps_fna": OUT / "refs/genomes/uhvdb.species_reps.fna.gz",
    "uhvdb_syldb": OUT / "refs/uhvdb.syldb",
}

required_now = [
    "new_reps",
    "old_reps",
    "deacon_idx",
    "bacteria_syldb",
    "analyze_img",
    "sylph_img",
    "samples_tsv",
]
prep_outputs = ["species_reps_fna", "uhvdb_syldb"]

print("Required before submit:")
ok = True
for k in required_now:
    p = paths[k]
    exists = p.is_file() and p.stat().st_size > 0
    print(f"  [{'OK' if exists else 'MISSING'}] {k}: {p}")
    ok = ok and exists

print("\nAfter prepare_species_reps.sbatch:")
prep_ok = True
for k in prep_outputs:
    p = paths[k]
    exists = p.is_file() and p.stat().st_size > 0
    print(f"  [{'OK' if exists else 'PENDING'}] {k}: {p}")
    prep_ok = prep_ok and exists

print("\nReady for prepare submit:" if ok else "\nFix missing paths before submit")
print("Ready for analyze submit:" if (ok and prep_ok) else "Analyze submit blocked until prep finishes")


## 3. Submit helpers

Run these from a login node with Slurm access. Uncomment the `!sbatch` / `!./submit_analyze.sh` lines when ready.


In [ ]:
print("# 1) Build species-rep FASTA + sylph sketch")
print(f"sbatch {OUT / 'prepare_species_reps.sbatch'}")
print()
print("# 2) After prep completes, submit analyze array")
print(f"cd {OUT} && ./submit_analyze.sh 8")
print()
print("# Optional: watch queue")
print("squeue -u $USER")

# Uncomment to submit:
# !sbatch {OUT / 'prepare_species_reps.sbatch'}
# !cd {OUT} && ./submit_analyze.sh 8


## 4. Results inventory


In [ ]:
results_dir = OUT / "results"
samples = []
if SAMPLES_TSV.is_file():
    with SAMPLES_TSV.open() as fh:
        reader = csv.DictReader(fh, delimiter="\t")
        samples = [row["sample"] for row in reader]

have_profile, have_depth, missing = [], [], []
for sample in samples:
    prof = results_dir / sample / f"{sample}.profile.tsv"
    depth = results_dir / sample / f"{sample}.depth.tsv.gz"
    if prof.is_file() and prof.stat().st_size > 0:
        have_profile.append(sample)
        if depth.is_file() and depth.stat().st_size > 0:
            have_depth.append(sample)
    else:
        missing.append(sample)

print(f"samples: {len(samples)}")
print(f"with profile: {len(have_profile)}")
print(f"with depth:   {len(have_depth)}")
print(f"missing:      {len(missing)}")
if missing[:20]:
    print("missing (first 20):", missing[:20])

jobid_file = OUT / "logs/analyze_jobid.txt"
if jobid_file.is_file():
    print("last analyze jobid:", jobid_file.read_text().strip())
